# Resumen documento

El objetivo de este código es que sea aplicable para cualquier geometría. Se establecen los parámetros A, B, C y se comprueban que son correctos. Dada la geometría también se obtienen los $\Delta_{ij}$, $d_{ij}$.
Lo que hay que cambiar para cada problema en específico es:

-Paso 1: Definición del grafo.  edges = {}...}

-Paso 5: Geometría ajustada con las distancias:  coordenadas = [...]

In [28]:
from bloqade.analog import start, cast, load, save
import os
import matplotlib.pyplot as plt
from bokeh.io import output_notebook
from bloqade.analog.atom_arrangement import ListOfLocations, Lieb, Square, Chain, Honeycomb, Kagome, Triangular, Rectangular
from bloqade.analog import piecewise_linear, cast
import numpy as np

output_notebook()

if not os.path.isdir("data"):
    os.mkdir("data")

Loading BokehJS ...

# Condiciones para la simulación

## Constantes $A,\space B, \space C$

$$
\begin{cases}
    A>(\Delta-2)B \\
    B>C
\end{cases}
$$

Extra para imponer bien el bloqueo de Rydberg: $$A\gg B$$

## Parámetros Hardware: $\Delta_{ij}, \space d_{ij}$

### Términos lineales: $\Delta_{ij}$

$$
\begin{equation}
    \Delta_i = - B\sum_{e\in E}(\deg(u)+\deg(v))x_e + B|E| + C\sum_e x_e
\end{equation}
$$

### Términos no lineales: $d_{ij}$

Tomamos $A \gg B \implies W_{ij} \simeq A$. Pero si no tendríamos $W_{12}=A+2B \quad ; \quad W_{13}=B$.

$$
\begin{cases}
    d_{ij} = \left( \frac{C_6}{A} \right) ^{1/6} \\
    A = \frac{C_6}{d_{ij}}
\end{cases}
$$

Además, para imponer bloqueo: $d_{ij}<R_B \implies \Omega<A = \frac{C_6}{d_{ij}}$



## Parámetros de simulación: $\Omega$, $T$

### Condiciones de la frecuencia de Rabi ($\Omega$)

1. Condición de bloqueo:
   Para asegurar el bloqueo de Rydberg: $V_{ij} > \Omega$
   $$d < R_B \iff \left( \frac{C_6}{A} \right)^{1/6} < \left( \frac{C_6}{\Omega}\right)^{1/6} \iff \Omega < A$$

2. Condición adiabática:
   Para que el sistema evolucione lentamente, el término $\Delta_i$ debe dominar sobre $\Omega$:
   $$\Omega \lesssim \min_{i}(\Delta_{i}^{\text{local}})$$

   Y para mantener la evolución adiabática, el tiempo de simulación ($T ==$  `sweep_time`) debe ser:
   $$T \gg \frac{1}{\delta E_{\text{min}}^2} \sim \frac{1}{\Omega^2}$$

3. Límites del hardware (Aquila):
   No los hago caso, pero por tener una referencia
   * Rango de frecuencia: $0 \leq \Omega \leq 15.8 \text{ rad/µs}$
   * Valor máximo: $2.5 \text{ MHz}$
   * Slew rate máximo: $250.0 \text{ rad/µs}^2$ ó $40.0 \text{ MHz/µs}$

# Parte específica de cada problema (definir el grafo)

Definimos las aristas (`edges`, se usa en Paso 1) y la geomtría de los átomos para la simulación (`coordenadas`, se usa en Paso 5). De paso se hacen algunas comprobaciones para que sea correcto

## Grafo 1: Grafo en Y
Solución válida para A=50, B=5, C_cost=0.8*B, omega_max=5.5

In [36]:
# ══════════════════════════════════════════════════════════════
# PASO 0: Definición del grafo
# ══════════════════════════════════════════════════════════════


# (Se usa en Paso 1). Definimos las aristas del grafo original (pares de vértices)
edges = {
    'e1': ('A1', 'A2'),
    'e2': ('A2', 'A3'),
    'e3': ('A3', 'A4'),
    'e4': ('A3', 'A5')
}

# (Se usa en el paso 5): Coordenadas de los átomos

coordenadas = [
    (-2*d, 0.0),                                  # Átomo 0 (e0)    
    (-d, 0.0),                                     # Átomo 1 (e1 - Centro)
    (0.0, 0.0),                                   # Átomo 2 (e2 - Centro)
    (d * np.cos(np.pi/3), d * np.sin(np.pi/3)),   # Átomo 3 (e3)
    (d * np.cos(-np.pi/3), d * np.sin(-np.pi/3))  # Átomo 4 (e4)
]


Coord = [(8.44,  3.90), (1.11,  3.78), (-5.18,  0.02), (-5.30,  7.34)]


geometria = ListOfLocations(Coord)
geometria.show()

assert len(Coord) == len(edges), \
    f"Error: No hay el mismo número de coordenadas ({len(coordenadas)}) que de aristas ({len(edges)}). Revisar edges (paso 1) y coordenadas (paso 5)."

## Grafo 2: Red 4x2
Una solución: 
A = 100, B = 5, C_cost = 0.1
omega_max = 10
También: A = 100, B = 7, C = 0.1, omega_max = 55

In [38]:
# ══════════════════════════════════════════════════════════════
# PASO 0: Definición del grafo
# ══════════════════════════════════════════════════════════════

# Auxiliar para mostrar la geometría del grafo original
#geometria_grafo_original = ListOfLocations([ ])
#geometria_grafo_original.show()

# (Se usa en Paso 1). Definimos las aristas del grafo original (pares de vértices)
edges = {
    'e0': ('A1', 'A2'),
    'e_11': ('A1', 'B1'),
    'e_12': ('A2', 'B2'),
    'e2': ('B1', 'B2'),
    'e_21': ('B1', 'C1'),
    'e_22': ('B2', 'C2'),
    'e3': ('C1', 'C2'),
    'e_31': ('C1', 'D1'),
    'e_32': ('C2', 'D2'),
    'e4': ('D1', 'D2'),
}

# (Se usa en el paso 5): Coordenadas de los átomos
sep = 1.2 * R_B  # separación vertical mínima entre filas > R_B


coordenadas = [
    (0.0, 0.0),
    (d * np.cos(np.pi/4),  d * np.sin(np.pi/4)),
    (d * np.cos(np.pi/4),  - d * np.sin(np.pi/4)),
    (2 * d * np.cos(np.pi/4), 0.0),
    (2 * d * np.cos(np.pi/4) + d * np.cos(np.pi/4), d * np.sin(np.pi/4)),
    (2 * d * np.cos(np.pi/4) + d * np.cos(np.pi/4), - d * np.sin(np.pi/4)),
    (4 * d * np.cos(np.pi/4), 0.0),
    (4 * d * np.cos(np.pi/4) + d * np.cos(np.pi/4), d * np.sin(np.pi/4)),
    (4 * d * np.cos(np.pi/4) + d * np.cos(np.pi/4), - d * np.sin(np.pi/4)),
    (6 * d * np.cos(np.pi/4), 0.0),
]


geometria = ListOfLocations(coordenadas)
geometria.show()

assert len(coordenadas) == len(edges), \
    f"Error: No hay el mismo número de coordenadas ({len(coordenadas)}) que de aristas ({len(edges)}). Revisar edges (paso 1) y coordenadas (paso 5)."#

# Grafo 3: Ciclo $C_6$
Solución: A = 100, B = 7, C = 0.1, omega_max = 55

In [31]:
edges = {
    'e0': ('A1', 'A2'),
    'e1': ('A2', 'A3'),
    'e2': ('A3', 'A4'),
    'e3': ('A4', 'A5'),
    'e4': ('A5', 'A6'),
    'e5': ('A6', 'A1'),
}

coordenadas = [
    ( d,                    0.0               ),  # e0
    ( d * np.cos(np.pi/3),  d * np.sin(np.pi/3)  ),  # e1
    (-d * np.cos(np.pi/3),  d * np.sin(np.pi/3)  ),  # e2
    (-d,                    0.0               ),  # e3
    (-d * np.cos(np.pi/3), -d * np.sin(np.pi/3)  ),  # e4
    ( d * np.cos(np.pi/3), -d * np.sin(np.pi/3)  ),  # e5
]


geometria = ListOfLocations(coordenadas)
geometria.show()

assert len(coordenadas) == len(edges), \
    f"Error: No hay el mismo número de coordenadas ({len(coordenadas)}) que de aristas ({len(edges)}). Revisar edges (paso 1) y coordenadas (paso 5)."

# Grafo 4: "Estrella" con 5 puntas
Solución: A = 100, B = 7 (Varía mucho la solución con esto), C = 0.1, omega_max = 55

In [32]:
edges = {
    'e_c': ('C', 'V0'),  # Átomo central en (0,0)
    'e0':  ('C', 'V1'),  # Átomo en el perímetro 
    'e1':  ('C', 'V2'),  # Átomo en el perímetro
    'e2':  ('C', 'V3'),  # Átomo en el perímetro
    'e3':  ('C', 'V4'),  # Átomo en el perímetro
    'e4':  ('C', 'V5')   # Átomo en el perímetro
}

r=d
coordenadas = [
    (0,0),
    ( r,                       0.0                      ),  # e0
    ( r * np.cos(2*np.pi/5),   r * np.sin(2*np.pi/5)   ),  # e1
    ( r * np.cos(4*np.pi/5),   r * np.sin(4*np.pi/5)   ),  # e2
    ( r * np.cos(6*np.pi/5),   r * np.sin(6*np.pi/5)   ),  # e3
    ( r * np.cos(8*np.pi/5),   r * np.sin(8*np.pi/5)   ),  # e4
]



geometria = ListOfLocations(coordenadas)
geometria.show()

assert len(coordenadas) == len(edges), \
    f"Error: No hay el mismo número de coordenadas ({len(coordenadas)}) que de aristas ({len(edges)}). Revisar edges (paso 1) y coordenadas (paso 5)."

# Parte genérica de la emulación
Una vez definido el grafo, esto vale para cualquiera (a falta de revisar las variables)

In [39]:
import itertools
from IPython.display import display, HTML # para mostrar mensajes de error en rojo y negrita

# ══════════════════════════════════════════════════════════════
# PASO 1: Análisis del grafo
# ══════════════════════════════════════════════════════════════
edges = edges # Definido en el paso 0
edge_list = list(edges.keys())  # ['e0', 'e1', 'e2', 'e3'...]

# Grado de cada vértice
from collections import defaultdict
degree = defaultdict(int)
for u, v in edges.values():
    degree[u] += 1
    degree[v] += 1

delta_grafo_max = max(degree.values())
print("── PASO 1: Análisis del grafo ──")
#print(f"  Aristas : {edges}")
#print(f"  Grados  : {dict(degree)}")
print(f"  Δ (grado máximo) = {delta_grafo_max}")




# ══════════════════════════════════════════════════════════════
# PASO 2: Elección de A, B, C respetando la jerarquía
# ══════════════════════════════════════════════════════════════
# Condición: A > (Δ-2)B  y  B > C > 0
# Con Δ=3 → A > B > C
A = 50  # Mayor A → más difícil colorear/excitar aristas contiguas (físicamente hay menor distancia entre átomos con interacción). Una A muy grande hace más demandante la simulación y tarda más
B = 5   # Mayor B → más aristas coloreadas/excitadas
C_cost = 0.1*B   # Mayor C_cost → menos aristas coloreadas
omega_max = 5.5 # Mayor Ω → menor R_B 

print(f"\n── Paso 2: Valores A, B, C ──")
print(f"  A={A}, B={B}, C={C_cost}")
if A > (delta_grafo_max - 2) * B and B > C_cost > 0:
    print(f"  Condiciones cumplidas (Δ={delta_grafo_max}):")
    print(f"  1. A > (Δ-2)B → {A} > {(delta_grafo_max-2)*B:.2f} ✓")
    print(f"  2. B > C → {B} > {C_cost} ✓")
else:
    display(HTML("<span style='color: #FF0000; font-weight: bold; font-size: 1.2em;'>Condiciones A,B,C no cumplidas:</span>"))
    print(f"  A > (Δ-2)B → {A} > {(delta_grafo_max-2)*B:.2f} ")
    print(f"  B > C → {B} > {C_cost} ")




# ══════════════════════════════════════════════════════════════
# PASO 3: Coeficientes lineales → Detuning local Δᵢ
# ══════════════════════════════════════════════════════════════
#   → Δᵢ = B·(deg(u) + deg(v)) + B|E| − C 

print(f"\n── PASO 3: Detuning local ──")
delta_local = {}
for e, (u, v) in edges.items():
    coef_HB = B * (degree[u] + degree[v]-1)
    coef_HC = C_cost
    delta_local[e] = coef_HB - coef_HC
    print(f"  {e}=({u},{v}): deg({u})={degree[u]}, deg({v})={degree[v]}"
      f"  →  Δ_{e} = {B}·{degree[u]+degree[v]} + {B}·{len(edges)} − {C_cost} = {delta_local[e]:.3f}")






# ══════════════════════════════════════════════════════════════
# PASO 4: Cálculo distancias a partir de A
# ══════════════════════════════════════════════════════════════
C6 = 2 * np.pi * 862690  # [rad/µs · µm^6]

# Radio de bloqueo de Rydberg
R_B = (C6 / omega_max) ** (1/6)
print(f"\n── PASO 4: Cálculo distancias ──")
print(f"  R_B = (C6/Ω)^(1/6) = {R_B:.2f} µm")

# Distancia entre átomos con interacción (W_ij ≈ A)
d = (C6 / A) ** (1/6)
print(f"  d = (C6/A)^(1/6) = {d:.2f} µm")

# Cálculo potencial (solo es para comparar con el detuning local, no se usa)
V_ij = C6 / d**6
print(f"  Potencial V_ij = C6/d^6 = {V_ij:.2f} rad/µs (V_ij_proximos = A)")

# Verificación de bloqueo: d < R_B ↔ Ω < A
if d < R_B:
    print(f"  Condición de bloqueo: d={d:.2f} < R_B={R_B:.2f} ✓  (A={A} > Ω={omega_max})")
else:
    display(HTML("<span style='color: #FF0000; font-weight: bold;'>Bloqueo NO garantizado: d > R_B . Aumenta A o reduce Ω </span>"))
    print(f"  d={d:.2f} > R_B={R_B:.2f}  →  aumenta A o reduce Ω")



# ══════════════════════════════════════════════════════════════
# PASO 4.2: Cálculo de distancias exactas considerando términos de B
# ══════════════════════════════════════════════════════════════
print(f"\n── 4.2 NOTA EXTRA: Distancias exactas (W_ij = A + M·B)  ──")
print("Esta parte es opcional, muestra cómo el término añadido de B puede afecta a las distancias requeridas para el bloqueo.")
d_required_exact = {}
for e1, e2 in itertools.combinations(edge_list, 2):
    u1, v1 = edges[e1]
    u2, v2 = edges[e2]
    shared = {u1, v1} & {u2, v2}

    V_HA = A if shared else 0
    V_HB = B * len(shared)
    if not shared:
        for w in {u1, v1}:
            for x in {u2, v2}:
                if any(({w, x} <= {u, v}) for u, v in edges.values()):
                    V_HB += B
                    break

    V_ij_exact = V_HA + V_HB
    d_ij_exact = (C6 / V_ij_exact) ** (1/6) if V_ij_exact > 0 else np.inf
    d_required_exact[(e1, e2)] = d_ij_exact

    bloqueo = "✓" if d_ij_exact < R_B else "⚠"
    print(f"  ({e1},{e2}): W={V_HA}+{V_HB:.1f}={V_ij_exact:.2f}  →  d_exact={d_ij_exact:.2f} µm  "
        f"vs  d_aprox={d:.2f} µm  {bloqueo}")




# ══════════════════════════════════════════════════════════════
# PASO 5: Geometría ajustada con las distancias
# ══════════════════════════════════════════════════════════════
print(f"\n── PASO 5: Geometría ajustada con las distancias ──")
print(f"  Para escribir en la imagen: R_B = {R_B:.2f} µm // d = {d:.2f} µm")


coordenadas = coordenadas # Definido en el paso 0




# ══════════════════════════════════════════════════════════════
# PASO 6: Waveforms y secuencia Bloqade
# ══════════════════════════════════════════════════════════════
omega_max  = omega_max
sweep_time = 11.0
delta_end  = 1 * max(delta_local.values())  # Δ(T) = Δᵢ,ₘₐₓ > Δᵢ . Garantiza que el detuning global 
durations  = [0.8, sweep_time, 0.8]

rabi_amplitude_values = [0, omega_max, omega_max, 0.0] # Esto define Ω(t)
rabi_detuning_values  = [-delta_end, -delta_end, delta_end, delta_end] # Esto define Δ(t)
# Visualización de los waveforms
#ver_Omega_t = piecewise_linear(durations, rabi_amplitude_values)
#ver_Omega_t.show()
#ver_Delta_t = piecewise_linear(durations, rabi_detuning_values)
#ver_Delta_t.show()

delta_local_min = min(delta_local.values())
# Condición sweep_time: T >> 1/Ω²
T_min_estimado = 1 / omega_max**2
#Buscamos: sweep_time >> T_min_estimado  En Python no existe >>, usamos factor 10 como criterio

print(f"\n── PASO 6: Waveforms y secuencia Bloqade ──")
print(f"sweep_time = {sweep_time} µs")
print(f"durations (µs) : {durations}")

print(f"Condiciones omega_max. Ω = {omega_max} rad/µs :")
if omega_max < A:
    print(f"    1.  Ω={omega_max} < A={A} ✓ ")
else:
    display(HTML("<span style='color: #FF0000; font-weight: bold;'>Condición 1 NO cumplida: Ω debe ser menor que A para garantizar el bloqueo</span>"))
    print(f"1.  Ω={omega_max} >= A={A}  →  reduce Ω o aumenta A")

if omega_max < delta_local_min:
    print(f"    2.  Ω={omega_max} < min(Δᵢ)={delta_local_min:.3f} ✓ ")
else:
    display(HTML("<span style='color: #FF0000; font-weight: bold;'>Condición 2 NO cumplida: Ω debe ser menor que el mínimo Δᵢ para garantizar la selectividad</span>"))
    print(f"    2.  Ω={omega_max} >= min(Δᵢ)={delta_local_min:.3f}  →  reduce Ω o aumenta A/B/C")

if sweep_time > 10 * T_min_estimado:
    print(f"    3. T >> 1/Ω² → {sweep_time} >> {T_min_estimado:.4f} µs ✓")
else:
    display(HTML("<span style='color: #FF0000; font-weight: bold;'>Condición 3 NO cumplida: sweep_time puede ser insuficiente para garantizar la adiabaticidad</span>"))
    print(f"    3. T >> 1/Ω² → {sweep_time} >> {T_min_estimado:.4f} µs  →  aumenta sweep_time o reduce Ω")



print(f"delta_end: Δ = {delta_end:.3f} rad/µs. Es el valor del detuning global")

print(f"Escalas de detuning: Δᵢ")
# El hardware aplicar el detuning como Δᵢ(t) = escala_i * Δ(t), así que debemos calcular las "escalas_i"
# escala_i = Δᵢ / delta_end  (para que el detuning efectivo final sea Δᵢ)
escalas = [delta_local[e] / delta_end for e in edge_list]
for e, esc in zip(edge_list, escalas):
    print(f"  {e}: Δ={delta_local[e]:.3f}  →  escala = {esc:.4f}  →  Δ efectivo = {esc*delta_end:.3f}")
print(f"\n")



# Ejecutamos el programa
prog = (
    geometria
    .rydberg.rabi.amplitude.uniform.piecewise_linear(durations, rabi_amplitude_values)
    .detuning.scale(escalas).piecewise_linear(durations, rabi_detuning_values)
)

emu_prog = prog.bloqade.python().run(shots=10000)
print("--- Valores de energía de los estados del hamiltoniano. Calculados por el emulador (no sé quitarlos): ---")
emu_prog.report().show()


── PASO 1: Análisis del grafo ──
  Δ (grado máximo) = 3

── Paso 2: Valores A, B, C ──
  A=50, B=5, C=0.5
  Condiciones cumplidas (Δ=3):
  1. A > (Δ-2)B → 50 > 5.00 ✓
  2. B > C → 5 > 0.5 ✓

── PASO 3: Detuning local ──
  e0=(A1,A2): deg(A1)=2, deg(A2)=2  →  Δ_e0 = 5·4 + 5·10 − 0.5 = 14.500
  e_11=(A1,B1): deg(A1)=2, deg(B1)=3  →  Δ_e_11 = 5·5 + 5·10 − 0.5 = 19.500
  e_12=(A2,B2): deg(A2)=2, deg(B2)=3  →  Δ_e_12 = 5·5 + 5·10 − 0.5 = 19.500
  e2=(B1,B2): deg(B1)=3, deg(B2)=3  →  Δ_e2 = 5·6 + 5·10 − 0.5 = 24.500
  e_21=(B1,C1): deg(B1)=3, deg(C1)=3  →  Δ_e_21 = 5·6 + 5·10 − 0.5 = 24.500
  e_22=(B2,C2): deg(B2)=3, deg(C2)=3  →  Δ_e_22 = 5·6 + 5·10 − 0.5 = 24.500
  e3=(C1,C2): deg(C1)=3, deg(C2)=3  →  Δ_e3 = 5·6 + 5·10 − 0.5 = 24.500
  e_31=(C1,D1): deg(C1)=3, deg(D1)=2  →  Δ_e_31 = 5·5 + 5·10 − 0.5 = 19.500
  e_32=(C2,D2): deg(C2)=3, deg(D2)=2  →  Δ_e_32 = 5·5 + 5·10 − 0.5 = 19.500
  e4=(D1,D2): deg(D1)=2, deg(D2)=2  →  Δ_e4 = 5·4 + 5·10 − 0.5 = 14.500

── PASO 4: Cálculo distancias ──
  